In [1]:
!pip install bert-score
!pip install --upgrade torchao
!pip install --upgrade transformers

from bert_score import BERTScorer
import time
import torch
import pandas as pd
import difflib
import json
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 46.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_NAME = "hynky/czech_news_dataset_v2"
BASE_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
MODEL_PATHS = ["finetuned_3.5_0.8B_model", "finetuned_3.5_2.0B_model", "distilled_qwen_0.8B_finetuned", "distilled_qwen_0.8B_unfinetuned", BASE_MODEL_NAME]

SYSTEM_PROMPT = (
    "Jsi profesionální český novinář a editor. "
    "Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí dodaného "
    "novinového článku. Shrnutí by mělo obsahovat nejdůležitější fakta a "
    "nesmí si vymýšlet žádné nové informace."
)

MAX_NEW_TOKENS = [100, 200]

BERT_SCORE_MODEL = "ufal/robeczech-base"

In [4]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Load dataset

In [5]:
dataset = load_dataset(DATASET_NAME, split="train[:100]")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00011-c758fbca39d29e(…):   0%|          | 0.00/285M [00:00<?, ?B/s]

data/train-00001-of-00011-04ed5181478f78(…):   0%|          | 0.00/283M [00:00<?, ?B/s]

data/train-00002-of-00011-7bab2e4f395098(…):   0%|          | 0.00/290M [00:00<?, ?B/s]

data/train-00003-of-00011-4f3be47507ad60(…):   0%|          | 0.00/300M [00:00<?, ?B/s]

...

data/test-00000-of-00002-9f5fef591354e92(…):   0%|          | 0.00/178M [00:00<?, ?B/s]

data/test-00001-of-00002-be0d3a48e095e91(…):   0%|          | 0.00/188M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1641471 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/144836 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/144837 [00:00<?, ? examples/s]

### Load model

In [6]:
def get_model_dtype():
  return torch.float16 if DEVICE == "cuda" else torch.float32

def load_model(model_path):
  model_dtype = get_model_dtype()

  tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

  model = AutoModelForCausalLM.from_pretrained(
      model_path,
      torch_dtype=model_dtype,
      device_map="auto",
  )

  if tokenizer.pad_token_id is None:
      tokenizer.pad_token = tokenizer.eos_token

  model.eval()

  return model, tokenizer

In [7]:
def build_prompt(article):
    user_prompt = (
        "Přečti si následující článek a napiš jeho stručné shrnutí "
        "(maximálně 2 až 3 věty).\n\n"
        f"ČLÁNEK:\n{article}"
    )

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    return prompt

def generate_summary(model, tokenizer, article, max_tokens):
    full_prompt = build_prompt(article)
    device = next(model.parameters()).device

    model_inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    prompt_length = model_inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.8,
            repetition_penalty=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def prepare_evaluation_data(dataset, model, tokenizer, max_tokens):
    t = time.time()

    eval_data = []

    start_time = time.time()
    data_len = len(dataset)

    for idx, item in enumerate(dataset, 1):
        article = item["content"]
        reference = item.get("brief", "")

        if (reference is None) or (reference.strip() == ""):
            continue

        summary = generate_summary(model, tokenizer, article, max_tokens)

        print("Summary:", summary)
        print("Ground truth:", reference)

        eval_data.append(
            {
                "response": summary,
                "reference": reference,
            }
        )

        print(f"[{idx} / {data_len}] Prepared example")

    elapsed_time = time.time() - start_time
    print(f"Finished preparing {data_len} examples in {elapsed_time:.2f}s\n")
    return eval_data

In [13]:
for model_path in MODEL_PATHS:
  model, tokenizer = load_model(model_path)

  for max_new_tokens in MAX_NEW_TOKENS:
    print(f"Generating summaries for {model_path} with max_new_tokens = {max_new_tokens}")
    eval_data = prepare_evaluation_data(dataset, model, tokenizer, max_new_tokens)

    df = pd.DataFrame(eval_data)
    df.to_json(
        f"summaries/{max_new_tokens}_tokens/{model_path}.json",
        orient="records",
        indent=4,
        force_ascii=False,
    )

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Generating summaries for Qwen/Qwen3.5-0.8B with max_new_tokens = 100
Summary: <think>

</think>

V New Yorku byla spuštěna skleněná světelná koule o průměru 1,8 metrů, která se spouští po stožáru a následně přeměnila obrovský časový kotel na obrovskou světelnou kouli s vířícími konfety a záblesky. V Washingtonu u Lincolnova památníku prezident Bill Clinton přicházel a odešel
Ground truth: Ohňostroji, veselicemi a s jásotem přivítal svět příchod roku 2000. Jako poslední obyvatelé Země sledovali západ Slunce v roce 1999 obyvatelé tichomořského zámořského území USA, Americké Samoy. Slunce se naposledy v roce 1999 schovalo za duhovým horizontem v pátek 06:57 večer místního času (tj. v sobotu v 07:57 SEČ). Petardy a bengálské ohně usmrtily v Evropě šest lidí a desítky dalších zranily. Oslavy i přesto nepotvrdily chmurné představy policistů, kteří čekali daleko větší potíže a byli ve zvýšené pohotovosti ve všech evropských metropolích
[1 / 100] Prepared example
Summary: <think>

</think>

No

### Rouge evaluation implementation

In [14]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def rouge_n(ref_text: str, out_text: str, n: int = 1):
    ref_tokens = ref_text.lower().split()
    cand_tokens = out_text.lower().split()

    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))

    overlap = sum((ref_ngrams & cand_ngrams).values())
    ref_count = sum(ref_ngrams.values())
    out_count = sum(cand_ngrams.values())

    recall = overlap / ref_count
    precision = overlap / out_count

    f1 = 0

    if (recall + precision > 0):
        f1 = (2 * recall * precision / (recall + precision))

    return {
        "overlap": overlap,
        "recall": recall,
        "precision": precision,
        "f1": f1,
    }

def lcs_length(x_tokens, y_tokens):
    matcher = difflib.SequenceMatcher(None, x_tokens, y_tokens)
    match = matcher.find_longest_match(0, len(x_tokens), 0, len(y_tokens))
    return match.size

def rouge_l(ref_text: str, out_text: str):
    ref_tokens = ref_text.lower().split()
    cand_tokens = out_text.lower().split()

    lcs = lcs_length(ref_tokens, cand_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(cand_tokens)

    f1 = 0
    if (recall + precision > 0):
      f1 = (2 * recall * precision / (recall + precision))

    return {
        "lcs": lcs,
        "recall": recall,
        "precision": precision,
        "f1": f1,
    }

In [ ]:
# def combine_baseline_student_summaries(student_file, baseline_file):
#     with open(student_file, "r", encoding="utf-8") as f:
#         student_data = json.load(f)

#     with open(baseline_file, "r", encoding="utf-8") as f:
#         baseline_data = json.load(f)

#     combined_data = []

#     for student_item, base_item in zip(student_data, baseline_data):
#         if (base_item["reference"] != student_item["reference"]):
#             continue

#         combined_data.append({
#             "response": student_item["response"],
#             "reference": base_item["response"],
#         })

#     return combined_data

# combined_eval_data = combine_baseline_student_summaries("generated_summaries_student.json", "generated_summaries_qwen1.5b.json")
# df = pd.DataFrame(combined_eval_data)
# df.to_json(
#     "generated_summaries_student_x_qwen1.5b.json",
#     orient="records",
#     indent=4,
#     force_ascii=False,
# )

In [15]:
def evaluate_summaries(summaries):
    eval_data = []

    metrics = ["recall", "precision", "f1"]
    rouge_types = ["rouge-1", "rouge-2", "rouge-l"]
    totals = {rt: {m: 0.0 for m in metrics} for rt in rouge_types}

    with open(summaries, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        response = item.get("response")
        reference = item.get("reference")
        scores = {
                    "rouge-1": rouge_n(reference, response, n=1),
                    "rouge-2": rouge_n(reference, response, n=2),
                    "rouge-l": rouge_l(reference, response)
                }

        for rt in rouge_types:
            for m in metrics:
                totals[rt][m] += scores[rt][m]

        out = {
            "response": response,
            "reference": reference,
            **scores
            }

        eval_data.append(out)

    avg_obj = {rt: {m: (totals[rt][m] / len(eval_data)) for m in metrics} for rt in rouge_types}

    return eval_data, avg_obj

def save_evaluation_results(eval_data, average_scores, filename):
    output = {"eval_data": eval_data, "average_scores": average_scores}

    with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=4, ensure_ascii=False)

In [16]:
distilled_finetuned_100 = evaluate_summaries("summaries/100_tokens/distilled_qwen_0.8B_finetuned.json")
distilled_unfinetuned_100 = evaluate_summaries("summaries/100_tokens/distilled_qwen_0.8B_unfinetuned.json")
finetuned_0_8_100 = evaluate_summaries("summaries/100_tokens/finetuned_3.5_0.8B_model.json")
finetuned_2_0_100 = evaluate_summaries("summaries/100_tokens/finetuned_3.5_2.0B_model.json")
baseline_0_8_100 = evaluate_summaries("summaries/100_tokens/Qwen/Qwen3.5-0.8B.json")

save_evaluation_results(distilled_finetuned_100[0], distilled_finetuned_100[1], "evaluations/100_tokens/distilled_qwen_0.8B_finetuned_eval.json")
save_evaluation_results(distilled_unfinetuned_100[0], distilled_unfinetuned_100[1], "evaluations/100_tokens/distilled_qwen_0.8B_unfinetuned_eval.json")
save_evaluation_results(finetuned_0_8_100[0], finetuned_0_8_100[1], "evaluations/100_tokens/finetuned_3.5_0.8B_model_eval.json")
save_evaluation_results(finetuned_2_0_100[0], finetuned_2_0_100[1], "evaluations/100_tokens/finetuned_3.5_2.0B_model_eval.json")
save_evaluation_results(baseline_0_8_100[0], baseline_0_8_100[1], "evaluations/100_tokens/Qwen/Qwen3.5-0.8B_eval.json")

distilled_finetuned_200 = evaluate_summaries("summaries/200_tokens/distilled_qwen_0.8B_finetuned.json")
distilled_unfinetuned_200 = evaluate_summaries("summaries/200_tokens/distilled_qwen_0.8B_unfinetuned.json")
finetuned_0_8_200 = evaluate_summaries("summaries/200_tokens/finetuned_3.5_0.8B_model.json")
finetuned_2_0_200 = evaluate_summaries("summaries/200_tokens/finetuned_3.5_2.0B_model.json")
baseline_0_8_200 = evaluate_summaries("summaries/200_tokens/Qwen/Qwen3.5-0.8B.json")

save_evaluation_results(distilled_finetuned_200[0], distilled_finetuned_200[1], "evaluations/200_tokens/distilled_qwen_0.8B_finetuned_eval.json")
save_evaluation_results(distilled_unfinetuned_200[0], distilled_unfinetuned_200[1], "evaluations/200_tokens/distilled_qwen_0.8B_unfinetuned_eval.json")
save_evaluation_results(finetuned_0_8_200[0], finetuned_0_8_200[1], "evaluations/200_tokens/finetuned_3.5_0.8B_model_eval.json")
save_evaluation_results(finetuned_2_0_200[0], finetuned_2_0_200[1], "evaluations/200_tokens/finetuned_3.5_2.0B_model_eval.json")
save_evaluation_results(baseline_0_8_200[0], baseline_0_8_200[1], "evaluations/200_tokens/Qwen/Qwen3.5-0.8B_eval.json")

#student_x_baseline_eval = evaluate_summaries("generated_summaries_student_x_qwen1.5b.json")

### BERTscore

In [17]:
scorer = BERTScorer(lang="cs", rescale_with_baseline=False, model_type=BERT_SCORE_MODEL, num_layers=9)

def eval_bertscore(summaries):
  responses = []
  references = []
  out_data = []

  total_precision = 0
  total_recall = 0
  total_f1 = 0

  with open(summaries, 'r', encoding='utf-8') as f:
    data = json.load(f)

  for item in data:
        response = item.get("response")
        reference = item.get("reference")

        responses.append(response)
        references.append(reference)

  P, R, F1 = scorer.score(responses, references)

  for i in range(len(responses)):
    p = P[i].item()
    r = R[i].item()
    f1 = F1[i].item()

    out = {
      "response": responses[i],
      "reference": references[i],
      "precision": p,
      "recall": r,
      "f1": f1,
    }

    out_data.append(out)

    total_precision += p
    total_recall += r
    total_f1 += f1

  out_len = len(out_data)

  avg_obj = {
      "precision": total_precision / out_len,
      "recall": total_recall / out_len,
      "f1": total_f1 / out_len,
  }

  return out_data, avg_obj

config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ufal/robeczech-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
distilled_finetuned_100_bertscore = eval_bertscore("summaries/100_tokens/distilled_qwen_0.8B_finetuned.json")
distilled_unfinetuned_100_bertscore = eval_bertscore("summaries/100_tokens/distilled_qwen_0.8B_unfinetuned.json")
finetuned_0_8_100_bertscore = eval_bertscore("summaries/100_tokens/finetuned_3.5_0.8B_model.json")
finetuned_2_0_100_bertscore = eval_bertscore("summaries/100_tokens/finetuned_3.5_2.0B_model.json")
baseline_0_8_bertscore = eval_bertscore("summaries/100_tokens/Qwen/Qwen3.5-0.8B.json")

save_evaluation_results(distilled_finetuned_100_bertscore[0], distilled_finetuned_100_bertscore[1], "evaluations_bertscore/100_tokens/distilled_qwen_0.8B_finetuned_eval.json")
save_evaluation_results(distilled_unfinetuned_100_bertscore[0], distilled_unfinetuned_100_bertscore[1], "evaluations_bertscore/100_tokens/distilled_qwen_0.8B_unfinetuned_eval.json")
save_evaluation_results(finetuned_0_8_100_bertscore[0], finetuned_0_8_100_bertscore[1], "evaluations_bertscore/100_tokens/finetuned_3.5_0.8B_model_eval.json")
save_evaluation_results(finetuned_2_0_100_bertscore[0], finetuned_2_0_100_bertscore[1], "evaluations_bertscore/100_tokens/finetuned_3.5_2.0B_model_eval.json")
save_evaluation_results(baseline_0_8_bertscore[0], baseline_0_8_bertscore[1], "evaluations_bertscore/100_tokens/Qwen/Qwen3.5-0.8B_eval.json")

distilled_finetuned_200_bertscore = eval_bertscore("summaries/200_tokens/distilled_qwen_0.8B_finetuned.json")
distilled_unfinetuned_200_bertscore = eval_bertscore("summaries/200_tokens/distilled_qwen_0.8B_unfinetuned.json")
finetuned_0_8_200_bertscore = eval_bertscore("summaries/200_tokens/finetuned_3.5_0.8B_model.json")
finetuned_2_0_200_bertscore = eval_bertscore("summaries/200_tokens/finetuned_3.5_2.0B_model.json")
baseline_0_8_200_bertscore = eval_bertscore("summaries/200_tokens/Qwen/Qwen3.5-0.8B.json")

save_evaluation_results(distilled_finetuned_200_bertscore[0], distilled_finetuned_200_bertscore[1], "evaluations_bertscore/200_tokens/distilled_qwen_0.8B_finetuned_eval.json")
save_evaluation_results(distilled_unfinetuned_200_bertscore[0], distilled_unfinetuned_200_bertscore[1], "evaluations_bertscore/200_tokens/distilled_qwen_0.8B_unfinetuned_eval.json")
save_evaluation_results(finetuned_0_8_200_bertscore[0], finetuned_0_8_200_bertscore[1], "evaluations_bertscore/200_tokens/finetuned_3.5_0.8B_model_eval.json")
save_evaluation_results(finetuned_2_0_200_bertscore[0], finetuned_2_0_200_bertscore[1], "evaluations_bertscore/200_tokens/finetuned_3.5_2.0B_model_eval.json")
save_evaluation_results(baseline_0_8_200_bertscore[0], baseline_0_8_200_bertscore[1], "evaluations_bertscore/200_tokens/Qwen/Qwen3.5-0.8B_eval.json")